## Cell 1: Setup and Memory-Optimized Data Loading

In [ ]:
import polars as pl
import torch
from torch_geometric.data import HeteroData
import networkx as nx
import matplotlib.pyplot as plt
import gc

# 1. Define paths
PARQUET_PATH = "./processed/ml_ready_transactions.parquet"

# 2. Select a small subgraph (1-2 companies) for the prototype
TARGET_CIKS = [320193, 789019] # Example CIKs (e.g., Apple, Microsoft)

print(f"Executing lazy query for issuerCiks: {TARGET_CIKS}")

# 3. Memory-Optimized Lazy Query
# We push filters down to the disk level so we only load a fraction of the data.
# We also downcast datatypes (e.g., Float64 -> Float32) to save 50% RAM per column.
lazy_q = (
    pl.scan_parquet(PARQUET_PATH)
    .filter(pl.col("issuerCik").is_in(TARGET_CIKS))
    .drop_nulls(subset=["transactionDate", "rptOwnerCik", "securityTitle", "issuerCik"])
    .with_columns([
        # Parse dates
        pl.col("transactionDate").str.strptime(pl.Datetime, format="%Y-%m-%d", strict=False),
        # Calculate Value and cast numeric columns to 32-bit to save RAM
        (pl.col("transactionShares") * pl.col("transactionPricePerShare")).cast(pl.Float32).alias("transactionValue"),
        pl.col("transactionShares").cast(pl.Float32),
        pl.col("transactionPricePerShare").cast(pl.Float32),
        pl.col("filingDate").cast(pl.Float32)
    ])
    .sort("transactionDate") # Sort on disk before collecting
)

# Collect into RAM
df = lazy_q.collect()
print(f"Successfully loaded {df.height} rows into memory.")
print(f"Estimated RAM usage of DataFrame: {df.estimated_size('mb'):.2f} MB")

## Cell 2: Temporal Feature Engineering (Leakage-Free)

In [ ]:
# Sort by Insider and Date to calculate historical features safely
df = df.sort(["rptOwnerCik", "transactionDate"])

# Calculate days since last trade (strictly relying on t-1)
df = df.with_columns([
    (pl.col("transactionDate") - pl.col("transactionDate").shift(1))
    .over("rptOwnerCik")
    .dt.total_days()
    .cast(pl.Float32) # Downcast to 32-bit float
    .fill_null(0.0) 
    .alias("days_since_last_trade")
])

# Re-sort strictly by time for the temporal graph
df = df.sort("transactionDate")
display(df.head(3))

## Cell 3: PyTorch Geometric Graph Construction

In [ ]:
data = HeteroData()

# 1. Create mapping dictionaries (Strings/Raw IDs to continuous integers 0...N)
# Using dictionaries is memory efficient for small/medium subgraphs.
unique_insiders = df["rptOwnerCik"].unique().to_list()
unique_companies = df["issuerCik"].unique().to_list()
unique_securities = df["securityTitle"].unique().to_list()

insider_map = {idx: i for i, idx in enumerate(unique_insiders)}
company_map = {idx: i for i, idx in enumerate(unique_companies)}
security_map = {idx: i for i, idx in enumerate(unique_securities)}

data['Insider'].num_nodes = len(unique_insiders)
data['Company'].num_nodes = len(unique_companies)
data['Security'].num_nodes = len(unique_securities)

# 2. Build Static Edge: Insider -> associated_with -> Company
df_assoc = df.unique(subset=["rptOwnerCik", "issuerCik"], keep="first")
src_assoc = [insider_map[x] for x in df_assoc["rptOwnerCik"].to_list()]
dst_assoc = [company_map[x] for x in df_assoc["issuerCik"].to_list()]

# PyG requires edge_index to be int64 (torch.long), but features can be float32
data['Insider', 'associated_with', 'Company'].edge_index = torch.tensor([src_assoc, dst_assoc], dtype=torch.long)
data['Insider', 'associated_with', 'Company'].edge_attr = torch.tensor(
    df_assoc.select(["isDirector", "isOfficer", "isTenPercentOwner"]).to_numpy(), 
    dtype=torch.float32 # 32-bit floats
)

# 3. Build Static Edge: Company -> issues -> Security
df_issues = df.unique(subset=["issuerCik", "securityTitle"], keep="first")
src_issues = [company_map[x] for x in df_issues["issuerCik"].to_list()]
dst_issues = [security_map[x] for x in df_issues["securityTitle"].to_list()]

data['Company', 'issues', 'Security'].edge_index = torch.tensor([src_issues, dst_issues], dtype=torch.long)

# 4. Build Temporal Edge: Insider -> trades -> Security
src_trades = [insider_map[x] for x in df["rptOwnerCik"].to_list()]
dst_trades = [security_map[x] for x in df["securityTitle"].to_list()]
timestamps = df["transactionDate"].dt.timestamp("ms").to_list()

data['Insider', 'trades', 'Security'].edge_index = torch.tensor([src_trades, dst_trades], dtype=torch.long)
data['Insider', 'trades', 'Security'].time = torch.tensor(timestamps, dtype=torch.long)

data['Insider', 'trades', 'Security'].edge_attr = torch.tensor(
    df.select(["transactionShares", "transactionPricePerShare", "transactionValue", "filingDate", "days_since_last_trade"]).fill_null(0.0).to_numpy(),
    dtype=torch.float32 # Force 32-bit floats to save memory
)

print(data)

# FREE UP RAM: Delete DataFrames and trigger garbage collection once PyG object is built
del df, df_assoc, df_issues
gc.collect()

## Cell 4: Visualization

In [ ]:
G = nx.DiGraph()

# Limit visualization to prevent Jupyter from freezing (rendering graphs is RAM intensive)
vis_limit = 15

for i in range(min(vis_limit, len(src_assoc))):
    insider = f"Insider_{src_assoc[i]}"
    company = f"Company_{dst_assoc[i]}"
    G.add_node(insider, type='Insider', color='lightblue')
    G.add_node(company, type='Company', color='lightgreen')
    G.add_edge(insider, company, label='associated_with')

for i in range(min(vis_limit, len(src_issues))):
    company = f"Company_{src_issues[i]}"
    security = f"Security_{dst_issues[i]}"
    G.add_node(company, type='Company', color='lightgreen')
    G.add_node(security, type='Security', color='salmon')
    G.add_edge(company, security, label='issues')

for i in range(min(vis_limit, len(src_trades))):
    insider = f"Insider_{src_trades[i]}"
    security = f"Security_{dst_trades[i]}"
    if G.has_node(insider) and G.has_node(security):
        G.add_edge(insider, security, label='trades')

colors = [nx.get_node_attributes(G, 'color').get(node, 'gray') for node in G.nodes()]
pos = nx.spring_layout(G, seed=42)

plt.figure(figsize=(10, 8))
nx.draw(G, pos, with_labels=True, node_color=colors, node_size=2000, font_size=9, font_weight='bold', edge_color='gray')
edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
plt.title("Temporal Heterogeneous Graph Structure (Sample)")
plt.show()